# Step 1: Data Cleaning — Kickstarter campaigns, 2016–2025

**Goal:** one row per finished Kickstarter campaign, with launch-day facts and two answer columns (`success`, `near_miss`).

**Source:** Web Robots monthly Kickstarter snapshots (https://webrobots.io/kickstarter-datasets/) — CSV from May 2019, JSON before that.

**Output:** `campaigns_clean.parquet` — 273,163 campaigns, 22 columns.

**To reproduce:** run only the cells marked PIPELINE, top to bottom (~1 hour). Cells marked EXPLORATION were checks used to make decisions.

**Team rule:** columns starting with `post_` describe what happened after launch. Never use them as model inputs.

### PIPELINE — Setup
Creates a temporary folder for the cleaned file and checks Colab's disk space and memory (12 GB, no backup memory — so we process one snapshot at a time).

In [2]:
import os

RAW_DIR = "/content"            # raw zips go here (temporary)
CLEAN_DIR = "/content/clean"    # cleaned file goes here (temporary until downloaded)
os.makedirs(CLEAN_DIR, exist_ok=True)

!df -h /content
!free -h

Filesystem      Size  Used Avail Use% Mounted on
overlay         108G   85M  108G   1% /
               total        used        free      shared  buff/cache   available
Mem:            12Gi       1.1Gi       9.1Gi       2.2Mi       2.7Gi        11Gi
Swap:             0B          0B          0B


### EXPLORATION — Inspect one snapshot (Oct 2026)
Looks at a single snapshot: 30 CSV pieces, ~92,000 rows, 42 columns. Shows the outcome labels and launch years.
Note: "1970" launch dates are placeholders for campaigns that never launched; they're removed later.
(Requires the Oct 2026 zip to be downloaded first.)

In [4]:
import pandas as pd, zipfile

z = zipfile.ZipFile(f"{RAW_DIR}/ks_2026-10-12.zip")
parts = [n for n in z.namelist() if n.endswith(".csv")]
df = pd.concat((pd.read_csv(z.open(n), low_memory=False) for n in parts), ignore_index=True)

print(len(parts), "CSV parts,", df.shape, "(rows, columns)")
print(df.columns.tolist())
print(df["state"].value_counts())
print(pd.to_datetime(df["launched_at"], unit="s").dt.year.value_counts().sort_index())
print(round(df.memory_usage(deep=True).sum() / 1e6), "MB in memory")

30 CSV parts, (91931, 42) (rows, columns)
['backers_count', 'blurb', 'category', 'converted_pledged_amount', 'country', 'country_displayable_name', 'created_at', 'creator', 'currency', 'currency_symbol', 'currency_trailing_code', 'current_currency', 'deadline', 'disable_communication', 'fx_rate', 'goal', 'id', 'is_disliked', 'is_in_post_campaign_pledging_phase', 'is_launched', 'is_liked', 'is_starrable', 'launched_at', 'location', 'name', 'percent_funded', 'photo', 'pledged', 'prelaunch_activated', 'profile', 'slug', 'source_url', 'spotlight', 'staff_pick', 'state', 'state_changed_at', 'static_usd_rate', 'urls', 'usd_exchange_rate', 'usd_pledged', 'usd_type', 'video']
state
successful    59034
failed        16993
submitted      7839
live           3543
canceled       2304
started        2212
suspended         6
Name: count, dtype: int64
launched_at
1970    10057
2009       11
2010       66
2011      422
2012     1181
2013     1508
2014     4217
2015     5390
2016     3795
2017     4064

### EXPLORATION — Success rate by year (one snapshot)
Finished campaigns per launch year. One snapshot looked suspiciously success-heavy (78%), which led us to combine many snapshots.

In [5]:
fin = df[df["state"].isin(["successful", "failed"])].copy()
fin["year"] = pd.to_datetime(fin["launched_at"], unit="s").dt.year

t = pd.crosstab(fin["year"], fin["state"])
t["total"] = t["successful"] + t["failed"]
t["success_rate"] = (t["successful"] / t["total"]).round(2)
print(t)

print(fin.loc[fin["state"] == "failed", "percent_funded"].describe())

state  failed  successful  total  success_rate
year                                          
2009        2           9     11          0.82
2010       21          45     66          0.68
2011       55         367    422          0.87
2012      108        1072   1180          0.91
2013      167        1334   1501          0.89
2014     1498        2506   4004          0.63
2015     2565        2522   5087          0.50
2016     1590        1997   3587          0.56
2017     1413        2488   3901          0.64
2018     1070        2594   3664          0.71
2019      944        2523   3467          0.73
2020      713        2090   2803          0.75
2021      574        2661   3235          0.82
2022      781        2794   3575          0.78
2023      820        3256   4076          0.80
2024     1673        8101   9774          0.83
2025     1843       11351  13194          0.86
2026     1156       11324  12480          0.91
count    16993.000000
mean         9.479058
std         15.9

### EXPLORATION — Near-miss count (old 70–99% definition)
With one snapshot, 2025 had only 37 near-misses — too few to test reliably. Superseded: near-miss is now 50–99% and computed later.

In [6]:
fin["near_miss"] = ((fin["state"] == "failed") & (fin["percent_funded"] >= 70)).astype(int)
print(fin[fin["year"] >= 2014].groupby("year")["near_miss"].sum())

year
2014    10
2015     9
2016     8
2017    12
2018    14
2019     9
2020     8
2021    10
2022     8
2023    21
2024    32
2025    37
2026    22
Name: near_miss, dtype: int64


### EXPLORATION — Does a second snapshot add new campaigns?
The Dec 2025 snapshot had 130,852 finished campaigns missing from Oct 2026. Conclusion: each snapshot is a partial sample, so we combine all of them.

In [7]:
!curl -s -o {RAW_DIR}/ks_2025-12-18.zip https://s3.amazonaws.com/weruns/forfun/Kickstarter/Kickstarter_2025-12-18T03_20_24_296Z.zip

z2 = zipfile.ZipFile(f"{RAW_DIR}/ks_2025-12-18.zip")
parts2 = [n for n in z2.namelist() if n.endswith(".csv")]
d2 = pd.concat((pd.read_csv(z2.open(n), low_memory=False, usecols=["id", "state", "percent_funded"])
                for n in parts2), ignore_index=True)

d2_fin = d2[d2["state"].isin(["successful", "failed"])].drop_duplicates("id")
added = d2_fin[~d2_fin["id"].isin(fin["id"])]
print(len(d2_fin), "finished campaigns in Dec 2025 snapshot")
print(len(added), "of them are NOT in the Oct 2026 snapshot")
print(added["state"].value_counts())
print((added["percent_funded"].between(70, 99.99) & (added["state"] == "failed")).sum(), "new near-misses")

188954 finished campaigns in Dec 2025 snapshot
130852 of them are NOT in the Oct 2026 snapshot
state
successful    75036
failed        55816
Name: count, dtype: int64
738 new near-misses


### PIPELINE — Get the list of snapshot links
Reads the Web Robots page and collects every CSV download link.

In [9]:
import re, requests

html = requests.get("https://webrobots.io/kickstarter-datasets/").text
urls = sorted(set(re.findall(r'https://s3\.amazonaws\.com/weruns/forfun/Kickstarter/Kickstarter_[^"\s]+\.zip', html)))

print(len(urls), "CSV snapshots found")
print("Oldest:", urls[0])
print("Newest:", urls[-1])

98 CSV snapshots found
Oldest: https://s3.amazonaws.com/weruns/forfun/Kickstarter/Kickstarter_2014-04-22.json.zip
Newest: https://s3.amazonaws.com/weruns/forfun/Kickstarter/Kickstarter_2026-10-12T07_08_07_724Z.zip


### PIPELINE — Choose CSV snapshots to process
Keeps snapshots from 2016 onward. In practice CSVs start in May 2019 (90 files).

In [10]:
run_urls = [u for u in urls if re.search(r"Kickstarter_(\d{4}-\d{2}-\d{2})", u).group(1) >= "2016-01-01"]
print(len(run_urls), "snapshots to process:", run_urls[0][-35:], "→", run_urls[-1][-35:])

90 snapshots to process: tarter_2019-05-16T03_20_20_822Z.zip → tarter_2026-10-12T07_08_07_724Z.zip


### PIPELINE — CSV combining function (+ test on 3 files)
For each snapshot: download → keep finished campaigns (successful/failed) launched before 2026 and only the columns in KEEP → add to the running table → keep the latest row per campaign `id` → delete the zip.
Removing duplicates after every file keeps memory small (~25 million raw rows collapse to a few hundred thousand campaigns).
The May 2019 CSV failed to download (server error); it's covered by its JSON version later.

In [11]:
import os, subprocess, zipfile, pandas as pd

KEEP = ["id", "state", "launched_at", "deadline", "created_at", "goal", "currency",
        "static_usd_rate", "usd_exchange_rate", "fx_rate", "country", "category",
        "name", "blurb", "staff_pick", "spotlight", "video", "location",
        "percent_funded", "pledged", "usd_pledged", "backers_count", "state_changed_at"]
CUTOFF = pd.Timestamp("2026-01-01").timestamp()   # drop campaigns launched in 2026

def save(df, name):
    out = df.copy()
    for c in out.select_dtypes("object").columns:
        out[c] = out[c].astype("string")          # avoids mixed-type save errors
    out.to_parquet(f"{CLEAN_DIR}/{name}", index=False)

def process(url_list):
    global combined
    for i, url in enumerate(url_list, 1):
        snap = re.search(r"Kickstarter_(\d{4}-\d{2}-\d{2})", url).group(1)
        path = f"{RAW_DIR}/snap.zip"
        try:
            subprocess.run(["curl", "-sf", "-o", path, url], check=True)
            z = zipfile.ZipFile(path)
            parts = [n for n in z.namelist() if n.endswith(".csv")]
            d = pd.concat((pd.read_csv(z.open(n), low_memory=False, usecols=lambda c: c in KEEP)
                           for n in parts), ignore_index=True)
            d = d[d["state"].isin(["successful", "failed"])].copy()
            d["id"] = pd.to_numeric(d["id"], errors="coerce")
            d = d[pd.to_numeric(d["launched_at"], errors="coerce") < CUTOFF]
            d["snapshot"] = snap
            before = len(combined)
            combined = (pd.concat([combined, d], ignore_index=True)
                          .sort_values("snapshot", kind="stable")
                          .drop_duplicates("id", keep="last"))
            print(f"{i}/{len(url_list)} {snap}: {len(d):,} finished, +{len(combined)-before:,} new → {len(combined):,} total")
        except Exception as e:
            print(f"{i}/{len(url_list)} {snap}: ERROR {e}")
        finally:
            if os.path.exists(path):
                os.remove(path)

combined = pd.DataFrame()
process(run_urls[:3])   # TEST RUN: first 3 snapshots only

1/3 2019-05-16: ERROR Command '['curl', '-sf', '-o', '/content/snap.zip', 'https://s3.amazonaws.com/weruns/forfun/Kickstarter/Kickstarter_2019-05-16T03_20_20_822Z.zip']' returned non-zero exit status 22.
2/3 2019-06-13: 194,963 finished, +171,098 new → 171,098 total
3/3 2019-07-18: 196,491 finished, +2,946 new → 174,044 total


### PIPELINE — Process all remaining CSV snapshots (~30 min)
Result: 333,468 campaigns. "+0 new" from April 2026 onward is expected — all 2025 campaigns had finished by then, and 2026 launches are excluded.
Note: several snapshots from mid-2021 to mid-2023 are very small (partial scrapes); later full snapshots recovered most campaigns.

In [12]:
import time
t0 = time.time()
process(run_urls[3:])
print(f"Done in {(time.time() - t0) / 60:.0f} minutes. Final total: {len(combined):,} campaigns")
save(combined, "combined_raw.parquet")

1/87 2019-08-15: 192,326 finished, +2,064 new → 176,108 total
2/87 2019-09-12: 197,612 finished, +2,211 new → 178,319 total
3/87 2019-10-17: 188,764 finished, +2,626 new → 180,945 total
4/87 2019-11-14: 194,752 finished, +2,406 new → 183,351 total
5/87 2019-12-12: 200,232 finished, +2,426 new → 185,777 total
6/87 2020-01-16: 196,730 finished, +2,145 new → 187,922 total
7/87 2020-02-13: 194,794 finished, +1,567 new → 189,489 total
8/87 2020-03-12: 199,821 finished, +2,383 new → 191,872 total
9/87 2020-04-16: 191,849 finished, +2,588 new → 194,460 total
10/87 2020-05-14: 203,353 finished, +1,533 new → 195,993 total
11/87 2020-06-18: 195,334 finished, +2,022 new → 198,015 total
12/87 2020-07-16: 201,586 finished, +1,996 new → 200,011 total
13/87 2020-08-13: 193,644 finished, +1,927 new → 201,938 total
14/87 2020-09-17: 204,293 finished, +2,666 new → 204,604 total
15/87 2020-10-15: 203,568 finished, +2,135 new → 206,739 total
16/87 2020-11-12: 197,842 finished, +2,350 new → 209,089 total
1

### EXPLORATION — Per-year check after CSVs
Superseded by the later per-year cell, which uses pledged ÷ goal and the 50–99% near-miss definition.

In [13]:
c = combined.copy()
c["year"] = pd.to_datetime(pd.to_numeric(c["launched_at"]), unit="s").dt.year
c["pf"] = pd.to_numeric(c["percent_funded"], errors="coerce")
c["near_miss"] = ((c["state"] == "failed") & (c["pf"] >= 70) & (c["pf"] < 100)).astype(int)

t = c.groupby("year").agg(
    campaigns=("id", "size"),
    success_rate=("state", lambda s: round((s == "successful").mean(), 2)),
    near_misses=("near_miss", "sum"))
print(t.loc[2014:])

!ls -lh {CLEAN_DIR}

      campaigns  success_rate  near_misses
year                                      
2014      24808          0.53           59
2015      34660          0.44           79
2016      26252          0.48           70
2017      25360          0.51           93
2018      25466          0.63           82
2019      28176          0.69           65
2020      25836          0.73           55
2021      23277          0.78           85
2022      20645          0.75          100
2023      21299          0.74          137
2024      26739          0.72          172
2025      26887          0.71          141
total 111M
-rw-r--r-- 1 root root 111M Oct 31 22:26 combined_raw.parquet


### PIPELINE — List JSON snapshots (Jan 2016 – May 2019)
Before May 2019, Web Robots only published JSON files. 40 found.

In [14]:
jurls = sorted(set(re.findall(
    r'https://s3\.amazonaws\.com/weruns/forfun/Kickstarter/Kickstarter_[^"\s]+\.json(?:\.gz|\.zip)', html)))
jold = [u for u in jurls if "2016-01-01" <= re.search(r"Kickstarter_(\d{4}-\d{2}-\d{2})", u).group(1) <= "2019-05-16"]
print(len(jold), "JSON snapshots, Jan 2016 – May 2019")
for u in jold[:2] + jold[-2:]:
    print("  ", u)

40 JSON snapshots, Jan 2016 – May 2019
   https://s3.amazonaws.com/weruns/forfun/Kickstarter/Kickstarter_2016-01-28T09_15_08_781Z.json.gz
   https://s3.amazonaws.com/weruns/forfun/Kickstarter/Kickstarter_2016-03-22T07_41_08_591Z.json.gz
   https://s3.amazonaws.com/weruns/forfun/Kickstarter/Kickstarter_2019-04-18T03_20_02_220Z.json.gz
   https://s3.amazonaws.com/weruns/forfun/Kickstarter/Kickstarter_2019-05-16T03_20_20_822Z.json.gz


### EXPLORATION — Peek inside one JSON file
Confirms the format: one campaign per line, with the same fields as the CSVs nested under "data".

In [15]:
import gzip
u = jold[-1]
p = f"{RAW_DIR}/test.json.gz" if u.endswith(".gz") else f"{RAW_DIR}/test.zip"
subprocess.run(["curl", "-sf", "-o", p, u], check=True)
!ls -lh {p}

if p.endswith(".gz"):
    with gzip.open(p, "rt") as f:
        print(f.read(1000))
else:
    z = zipfile.ZipFile(p)
    print(z.namelist()[:5])
    print(z.open(z.namelist()[0]).read(1000))

-rw-r--r-- 1 root root 169M Oct 31 22:34 /content/test.json.gz
{"table_id":"Kickstarter","robot_id":"Kickstarter","run_id":"Kickstarter_2019-05-16T03_20_20_822Z","data":{"id":177028447,"photo":{"key":"assets/011/281/287/db432df50103184abf64f9d151f8f994_original.jpg","full":"https://ksr-ugc.imgix.net/assets/011/281/287/db432df50103184abf64f9d151f8f994_original.jpg?ixlib=rb-2.0.0&crop=faces&w=560&h=315&fit=crop&v=1463680351&auto=format&frame=1&q=92&s=6b204806f22b079293f4eb57dceb178b","ed":"https://ksr-ugc.imgix.net/assets/011/281/287/db432df50103184abf64f9d151f8f994_original.jpg?ixlib=rb-2.0.0&crop=faces&w=352&h=198&fit=crop&v=1463680351&auto=format&frame=1&q=92&s=d49a51324ce8c8a6d93e9ff271ada9f2","med":"https://ksr-ugc.imgix.net/assets/011/281/287/db432df50103184abf64f9d151f8f994_original.jpg?ixlib=rb-2.0.0&crop=faces&w=272&h=153&fit=crop&v=1463680351&auto=format&frame=1&q=92&s=5ca6277f547cee9f61839dca7a27e2cd","little":"https://ksr-ugc.imgix.net/assets/011/281/287/db432df50103184abf64f

### PIPELINE — JSON reading function
Same logic as the CSV function. Nested fields (category, location) are converted to text so they match the CSV format.

In [16]:
import json, gzip, io

def lines_from(path):
    if path.endswith(".gz"):
        with gzip.open(path, "rt", encoding="utf-8") as f:
            yield from f
    else:
        z = zipfile.ZipFile(path)
        for n in z.namelist():
            with z.open(n) as f:
                yield from io.TextIOWrapper(f, encoding="utf-8")

def read_json_snapshot(path):
    rows = []
    for line in lines_from(path):
        line = line.strip()
        if not line:
            continue
        rec = json.loads(line).get("data", {})
        if rec.get("state") not in ("successful", "failed"):
            continue
        row = {k: rec.get(k) for k in KEEP}
        for k, v in row.items():
            if isinstance(v, (dict, list)):
                row[k] = json.dumps(v, separators=(",", ":"))   # match CSV text format
        rows.append(row)
    return pd.DataFrame(rows)

def process_json(url_list):
    global combined
    for i, url in enumerate(url_list, 1):
        snap = re.search(r"Kickstarter_(\d{4}-\d{2}-\d{2})", url).group(1)
        path = f"{RAW_DIR}/snap.json.gz" if url.endswith(".gz") else f"{RAW_DIR}/snap.zip"
        try:
            ids_before = set(combined["id"])
            subprocess.run(["curl", "-sf", "-o", path, url], check=True)
            d = read_json_snapshot(path)
            d["id"] = pd.to_numeric(d["id"], errors="coerce")
            d = d[pd.to_numeric(d["launched_at"], errors="coerce") < CUTOFF]
            d["snapshot"] = snap
            combined = (pd.concat([combined, d], ignore_index=True)
                          .sort_values("snapshot", kind="stable")
                          .drop_duplicates("id", keep="last"))
            new = combined[~combined["id"].isin(ids_before)]
            yrs = pd.to_datetime(pd.to_numeric(new["launched_at"]), unit="s").dt.year
            new_fail_1618 = ((new["state"] == "failed") & yrs.between(2016, 2018)).sum()
            print(f"{i}/{len(url_list)} {snap}: {len(d):,} finished, +{len(new):,} new "
                  f"({new_fail_1618:,} are 2016–18 failures) → {len(combined):,} total")
        except Exception as e:
            print(f"{i}/{len(url_list)} {snap}: ERROR {e}")
        finally:
            if os.path.exists(path):
                os.remove(path)

### PIPELINE — Test JSON on two files
Key finding: failed campaigns drop out of later snapshots over time. The Oct 2017 file recovered old failures that no 2019+ snapshot contained.
(This cell also defines `test`, which the full JSON run uses.)

In [17]:
print(len(jold), "JSON snapshots available")
test = [jold[-1], jold[len(jold) // 2]]    # May 2019 + one from the middle (~2017)
process_json(test)

40 JSON snapshots available


/tmp/ipykernel_4289/675116665.py:41: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined = (pd.concat([combined, d], ignore_index=True)


1/2 2019-05-16: 194,336 finished, +1,729 new (312 are 2016–18 failures) → 335,197 total


/tmp/ipykernel_4289/675116665.py:41: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined = (pd.concat([combined, d], ignore_index=True)


2/2 2017-10-15: 144,244 finished, +31,871 new (1,064 are 2016–18 failures) → 367,068 total


### EXPLORATION — What the JSON test added, and missing columns
JSON rows have no `percent_funded`, `usd_exchange_rate` or `video`. Fix: calculate percent funded as pledged ÷ goal, convert currency with `static_usd_rate`, and drop `video`.

In [18]:
# what did the 2017 file add, by launch year and outcome?
n17 = combined[combined["snapshot"] == "2017-10-15"]
yr = pd.to_datetime(pd.to_numeric(n17["launched_at"]), unit="s").dt.year
print(pd.crosstab(yr, n17["state"]))

# which columns are missing in the JSON rows?
js = combined[combined["snapshot"] < "2019-06-01"]
print(js.isna().mean().round(2).sort_values(ascending=False).head(10))

state        failed  successful
launched_at                    
2009            168         235
2010           1107        1874
2011           1766        4401
2012           2087        4375
2013           1902        2256
2014           5840        1058
2015           3688          18
2016            733          17
2017            331          15
usd_exchange_rate    1.00
percent_funded       1.00
video                1.00
fx_rate              0.95
location             0.01
backers_count        0.00
category             0.00
blurb                0.00
deadline             0.00
currency             0.00
dtype: float64


### PIPELINE — Process all JSON snapshots (~21 min)
Recovered ~10,500 failed campaigns from 2016–2018. Total: 437,547 campaigns (all launch years).

In [19]:
import warnings, time
warnings.filterwarnings("ignore", category=FutureWarning)   # hide the harmless warnings

t0 = time.time()
process_json([u for u in jold if u not in test])
print(f"Done in {(time.time() - t0) / 60:.0f} minutes. Total: {len(combined):,} campaigns")
save(combined, "combined_raw.parquet")    # overwrites the earlier save

1/38 2016-01-28: 132,853 finished, +24,275 new (10 are 2016–18 failures) → 391,343 total
2/38 2016-03-22: 136,937 finished, +7,207 new (612 are 2016–18 failures) → 398,550 total
3/38 2016-04-15: 139,022 finished, +1,759 new (403 are 2016–18 failures) → 400,309 total
4/38 2016-05-15: 131,044 finished, +2,027 new (558 are 2016–18 failures) → 402,336 total
5/38 2016-06-15: 144,010 finished, +1,643 new (519 are 2016–18 failures) → 403,979 total
6/38 2016-07-15: 146,242 finished, +884 new (528 are 2016–18 failures) → 404,863 total
7/38 2016-08-15: 148,379 finished, +720 new (460 are 2016–18 failures) → 405,583 total
8/38 2016-09-15: 150,139 finished, +599 new (354 are 2016–18 failures) → 406,182 total
9/38 2016-10-15: 151,739 finished, +800 new (349 are 2016–18 failures) → 406,982 total
10/38 2016-11-12: 153,454 finished, +708 new (381 are 2016–18 failures) → 407,690 total
11/38 2016-12-15: 155,707 finished, +786 new (387 are 2016–18 failures) → 408,476 total
12/38 2017-01-15: 157,743 finis

### Optional — Download a backup of the combined data (137 MB)

In [21]:
from google.colab import files
files.download(f"{CLEAN_DIR}/combined_raw.parquet")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### PIPELINE — Create the answer columns
- `pf` (percent funded) = pledged ÷ goal × 100 — matches Kickstarter's own value for 100% of CSV rows
- `success` = 1 if the campaign reached its goal
- `near_miss` = 1 if it failed but raised 50–99% of its goal (changed from 70–99% to get enough cases: 541 in 2024, 410 in 2025)

In [22]:
c = combined.copy()
c["year"] = pd.to_datetime(pd.to_numeric(c["launched_at"]), unit="s").dt.year
c["pf"] = pd.to_numeric(c["pledged"], errors="coerce") / pd.to_numeric(c["goal"], errors="coerce") * 100

# check the formula against Kickstarter's own percent_funded (CSV rows only)
both = c["percent_funded"].notna()
diff = (c.loc[both, "pf"] - pd.to_numeric(c.loc[both, "percent_funded"], errors="coerce")).abs()
print(f"Formula check: {(diff < 0.5).mean():.1%} of CSV rows match within 0.5 points")

c["success"] = (c["state"] == "successful").astype(int)
c["near_miss"] = ((c["state"] == "failed") & (c["pf"] >= 50) & (c["pf"] < 100)).astype(int)

t = c.groupby("year").agg(
    campaigns=("id", "size"),
    success_rate=("success", lambda s: round(s.mean(), 2)),
    near_misses=("near_miss", "sum"))
print(t.loc[2016:2025])
print((c.loc[c["success"] == 1, "pf"] < 100).sum(), "successes under 100% funded")

Formula check: 100.0% of CSV rows match within 0.5 points
      campaigns  success_rate  near_misses
year                                      
2016      36257          0.45          585
2017      34099          0.56          567
2018      29884          0.63          521
2019      28240          0.69          386
2020      25836          0.73          336
2021      23277          0.78          266
2022      20645          0.75          432
2023      21299          0.74          503
2024      26739          0.72          541
2025      26887          0.71          410
4 successes under 100% funded


### EXPLORATION — Choose the currency conversion
`static_usd_rate` is never missing and reproduces Kickstarter's own USD pledged amounts for 100% of rows, so we use it for goals too. 31.5% of campaigns are non-USD.

In [23]:
for col in ["static_usd_rate", "usd_pledged", "currency", "goal"]:
    print(f"{col}: {c[col].isna().mean():.2%} missing")

rate = pd.to_numeric(c["static_usd_rate"], errors="coerce")
usd_p = pd.to_numeric(c["usd_pledged"], errors="coerce")
calc = pd.to_numeric(c["pledged"], errors="coerce") * rate
match = ((calc - usd_p).abs() <= 0.01 * usd_p.abs() + 1)[usd_p.notna()]
print(f"pledged × static_usd_rate matches usd_pledged: {match.mean():.1%}")

nonusd = c["currency"] != "USD"
print(f"Non-USD campaigns: {nonusd.mean():.1%}")
print(c.loc[nonusd, ["currency", "goal", "static_usd_rate", "pledged", "usd_pledged"]].sample(5, random_state=1))

static_usd_rate: 0.00% missing
usd_pledged: 0.00% missing
currency: 0.00% missing
goal: 0.00% missing
pledged × static_usd_rate matches usd_pledged: 100.0%
Non-USD campaigns: 31.5%
       currency      goal static_usd_rate  pledged    usd_pledged
208236      SEK   25000.0        0.114462      0.0            0.0
354888      EUR    6000.0        1.141145     12.0      13.693737
44358       HKD  100000.0        0.128939   2500.0     322.346925
23378       CAD     100.0        0.806101   2002.0  1613.81464244
148902      SGD    3800.0        0.732554   5498.0    4027.579198


### PIPELINE — Convert goals to USD; test two suspicious columns
- `goal_usd` = goal × static_usd_rate; `log_goal_usd` puts goals on a log scale so huge goals don't dominate
- `spotlight`: 100% of spotlighted campaigns succeeded → it reveals the answer → dropped
- `staff_pick`: 91% vs 56% success, and can be assigned mid-campaign → kept for reference only (`post_staff_pick`)

In [24]:
import numpy as np

c["goal_usd"] = pd.to_numeric(c["goal"]) * pd.to_numeric(c["static_usd_rate"])
c["log_goal_usd"] = np.log1p(c["goal_usd"])     # log scale: goals range from $1 to millions
print(c["goal_usd"].describe().round(0))

for col in ["spotlight", "staff_pick"]:
    print(f"\nSuccess rate by {col}:")
    print(pd.crosstab(c[col].astype(str), c["state"], normalize="index").round(3))

count       437547.0
mean         30909.0
std         926983.0
min              0.0
25%           1300.0
50%           4600.0
75%          11729.0
max      152350076.0
Name: goal_usd, dtype: float64

Success rate by spotlight:
state      failed  successful
spotlight                    
False         1.0         0.0
True          0.0         1.0

Success rate by staff_pick:
state       failed  successful
staff_pick                    
False        0.445       0.555
True         0.089       0.911


### PIPELINE — Build, check and save the final file
Keeps launches 2016–2025, renames after-launch columns with a `post_` prefix, drops unused columns, checks for duplicate IDs (0), saves `campaigns_clean.parquet` (273,163 rows, 22 columns, 74 MB) and downloads it.

In [25]:
clean = c[c["year"].between(2016, 2025)].copy()
clean["usd_pledged"] = pd.to_numeric(clean["usd_pledged"], errors="coerce")

clean = clean.rename(columns={
    "state": "post_state", "usd_pledged": "post_usd_pledged", "pf": "post_pct_funded",
    "backers_count": "post_backers", "staff_pick": "post_staff_pick",
    "state_changed_at": "post_state_changed_at"})

cols = (["id", "year", "snapshot"]                                    # bookkeeping
        + ["success", "near_miss"]                                    # answers
        + ["log_goal_usd", "goal_usd", "category", "country", "currency",
           "launched_at", "deadline", "created_at", "name", "blurb", "location"]   # launch-day inputs
        + ["post_state", "post_usd_pledged", "post_pct_funded", "post_backers",
           "post_staff_pick", "post_state_changed_at"])               # after launch: never model inputs
clean = clean[cols]

print(clean.shape, "| duplicate ids:", clean["id"].duplicated().sum())
print(clean["year"].value_counts().sort_index())
save(clean, "campaigns_clean.parquet")
!ls -lh {CLEAN_DIR}
files.download(f"{CLEAN_DIR}/campaigns_clean.parquet")

(273163, 22) | duplicate ids: 0
year
2016    36257
2017    34099
2018    29884
2019    28240
2020    25836
2021    23277
2022    20645
2023    21299
2024    26739
2025    26887
Name: count, dtype: int64
total 211M
-rw-r--r-- 1 root root  74M Oct 31 23:17 campaigns_clean.parquet
-rw-r--r-- 1 root root 137M Oct 31 22:59 combined_raw.parquet


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Optional — Browse a 1,000-row sample as a table

In [26]:
from google.colab import data_table
data_table.enable_dataframe_formatter()
clean.sample(1000, random_state=1)

,id,year,snapshot,success,near_miss,log_goal_usd,goal_usd,category,country,currency,...,created_at,name,blurb,location,post_state,post_usd_pledged,post_pct_funded,post_backers,post_staff_pick,post_state_changed_at
295655,2048749321,2025,2026-08-12,1,0,3.258097,25.000000,"{""id"":34,""name"":""Tabletop Games"",""analytics_na...",US,USD,...,1751730622,Halloween Heists,Two Halloween-Themed Heists for TTRPG Spooky S...,"{""id"":2383489,""name"":""Colorado Springs"",""slug""...",successful,256.000000,1024.000000,15,False,1755266400
335530,865367612,2022,2026-09-10,1,0,8.517393,5000.000000,"{""id"":46,""name"":""Children's Books"",""analytics_...",US,USD,...,1641564283,A Tale of the Pirates Three,A children's book adventure,"{""id"":2487796,""name"":""San Antonio"",""slug"":""san...",successful,5006.000000,100.120000,52,False,1666536093
125159,52208450,2019,2020-06-18,1,0,8.987322,8000.000000,"{""id"":28,""name"":""Product Design"",""slug"":""desig...",US,USD,...,1549298463,Onli Travel Venture Rolling Pack: Never check ...,Modular 3-part luggage system that combines a ...,"{""id"":2367105,""name"":""Boston"",""slug"":""boston-m...",successful,61410.000000,767.625000,202,False,1554911783
147008,293584063,2017,2021-05-13,1,0,7.824446,2500.000000,"{""id"":46,""name"":""Children's Books"",""analytics_...",US,USD,...,1482794599,Grab The Wheel: Help Young Children Manage Exp...,The Play-Based Kit Teachers and Parents Can Us...,"{""id"":2442047,""name"":""Los Angeles"",""slug"":""los...",successful,3324.000000,132.960000,63,True,1496188801
316975,1396712384,2017,2026-09-10,0,0,9.305741,11000.000000,"{""id"":318,""name"":""Faith"",""analytics_name"":""Fai...",US,USD,...,1489703170,Aleluya (In honor of the late Leonard Cohen),Inspiring version of Hallelujah (Aleluya) in S...,"{""id"":2450022,""name"":""Miami"",""slug"":""miami-fl""...",failed,10.000000,0.090909,1,False,1491343201
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6606,1124270675,2016,2017-04-15,0,0,7.337200,1535.404900,"{""id"":35,""name"":""Video Games"",""slug"":""games/vi...",AU,AUD,...,1475133258,Character Development System for Unity®,Create amazing characters for Unity® right in ...,"{""id"":1103816,""name"":""Melbourne"",""slug"":""melbo...",failed,180.410076,11.750000,17,False,1481187475
248573,11487548,2021,2026-01-12,1,0,8.372711,4326.352095,"{""id"":46,""name"":""Children's Books"",""analytics_...",GB,GBP,...,1612867207,Always Here,A heartwarming book to teach our children that...,"{""id"":44418,""name"":""London"",""slug"":""london-gb""...",successful,4710.141394,108.870968,72,True,1617235200
195776,1209350968,2020,2024-04-15,1,0,8.415600,4515.983500,"{""id"":273,""name"":""Playing Cards"",""analytics_na...",HK,HKD,...,1600829049,Van Gogh Playing Cards,"""The unkindness of the world hurt him, but the...","{""id"":2165352,""name"":""Hong Kong"",""slug"":""hong-...",successful,19142.866972,423.891429,393,False,1602684057
249968,1468691576,2022,2026-02-12,0,0,9.210440,10000.000000,"{""id"":51,""name"":""Software"",""analytics_name"":""S...",US,USD,...,1638470021,A platform to help you improve EQ and relation...,A platform to help you increase success by dev...,"{""id"":2372044,""name"":""Burlingame"",""slug"":""burl...",failed,2.000000,0.020000,2,False,1644706645
